## **Christian EDA - working with the [Causer et al 2023 paper](https://doi.org/10.1038/s41698-023-00444-2).**

In [ ]:
import json
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import tifffile
from PIL import Image
from scipy import ndimage as ndi
from scipy.signal import fftconvolve
from scipy.spatial import cKDTree
from scipy.stats import binned_statistic_2d, spearmanr
from skimage.color import rgb2hed
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import closing, dilation, disk, opening, remove_small_objects
from skimage.registration._masked_phase_cross_correlation import cross_correlate_masked
from skimage.transform import warp
from sklearn.metrics import roc_auc_score

#### Loading in the Visium/Codex

In [ ]:
# Paths to data
DATA = Path("../../data/Causer_et_al_2023")
VISIUM = {"S01_MAR21": "Visium_S01_MAR21", "S01_SEP21": "Visium_S01_SEP21", "P5": "P5_upload"}
CODEX = DATA / "CODEX_S01"

###


# Loading in the seperate Visium files
def load_visium(folder, library_id):
    a = sc.read_10x_h5(folder / "filtered_feature_bc_matrix.h5")
    a.var_names_make_unique()
    pos = pd.read_csv(
        folder / "spatial/tissue_positions_list.csv",
        header=None,
        index_col=0,
        names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
    )
    a.obs = a.obs.join(pos, how="left")
    a.obsm["spatial"] = a.obs[["pxl_col", "pxl_row"]].to_numpy(float)
    sf = json.loads((folder / "spatial/scalefactors_json.json").read_text())
    images = {"hires": np.asarray(Image.open(folder / "spatial/tissue_hires_image.png"))}
    a.uns["spatial"] = {library_id: {"images": images, "scalefactors": sf}}
    a.obs["sample"] = library_id
    return a


adatas = {name: load_visium(DATA / folder, name) for name, folder in VISIUM.items()}

# Building shared Visium AnnData dictionary
vis = sc.concat(adatas, label="sample", uns_merge="unique", index_unique="-")
display(vis)  # 5566 spots × 17943 genes; vis.obs["sample"] says which section each spot came from

###

# Loading in the Codex
raw = pd.read_csv(
    CODEX / "measurement_cell_S01_2_8_9.tsv", sep="\t", dtype=str, keep_default_na=False
)
# print("raw rows:", len(raw)) #268489
is_header = raw["Image"].eq("Image")
# print("repeated header rows:", is_header.sum()) #4

# Remove the repeated header and duplicate cells
cells = raw[~is_header].drop_duplicates().reset_index(drop=True)
# print("unique cells:", len(cells)) #53697
meta_cols = ["Image", "Name", "Class", "Parent", "ROI"]
num_cols = [c for c in cells.columns if c not in meta_cols]
cells[num_cols] = cells[num_cols].apply(pd.to_numeric)
# print(cells[meta_cols].nunique()) #Image=1,Name = 1,Class=1,Parent=1,ROI=1
# print("any NaN in numeric cols:", cells[num_cols].isna().any().any()) #False

# Inspect what were used in Codex panel
markers = [
    c.removeprefix("Nucleus: ").removesuffix(" mean")
    for c in cells.columns
    if c.startswith("Nucleus: ") and c.endswith(" mean")
]
# print(markers)
# ['DAPI', 'CD8-BX026', 'CD3e-BX045', 'CD20-BX007', 'KI67-BX047', 'CD107a-BX006',
#'PANCK-BX019', 'CD45RO-BX017', 'HLA-DR-BX033', 'CD4-BX003', 'CD68-BX015', 'CD45-CX021']
# for comp in ["Nucleus", "Cytoplasm", "Cell"]:
# stats = sorted({c.split(f"{markers[1]} ")[-1] for c in cells.columns if c.startswith(f"{comp}: {markers[1]} ")})
# print(comp, stats)
# Nucleus ['max', 'mean', 'min', 'range', 'std dev', 'sum']
# Cytoplasm ['max', 'mean', 'min', 'std dev']
# Cell ['max', 'mean', 'min', 'std dev']
morph = [c for c in cells.columns if ":" in c and not any(m in c for m in markers)]
# print(morph)
#'Nucleus: Area', 'Nucleus: Perimeter', 'Nucleus: Circularity', 'Nucleus: Max caliper', 'Nucleus: Min caliper',
# 'Nucleus: Eccentricity', 'Cell: Area', 'Cell: Perimeter', 'Cell: Circularity', 'Cell: Max caliper', 'Cell: Min caliper', 'Cell: Eccentricity']

# Building Codex AnnData
X = cells[[f"Cell: {m} mean" for m in markers]].to_numpy(np.float32)
codex = ad.AnnData(
    X=X, obs=cells[morph].rename(columns=lambda c: c.replace(": ", "_").replace(" ", "_"))
)
codex.obs_names = [f"cell_{i}" for i in range(codex.n_obs)]
codex.var_names = [m.split("-BX")[0].split("-CX")[0] for m in markers]  # "CD8-BX026" → "CD8"
codex.var["panel_name"] = markers
for comp in ["Nucleus", "Cytoplasm"]:
    codex.layers[f"{comp.lower()}_mean"] = cells[[f"{comp}: {m} mean" for m in markers]].to_numpy(
        np.float32
    )
codex.obsm["spatial"] = cells[["Centroid X µm", "Centroid Y µm"]].to_numpy(float)

# Summary of Codex AnnData markers
# Smaller values as image is only 8-bit
# Large CD8/CD3e counts - either lots of T cells or antibody background is high
pd.DataFrame(codex.X, columns=codex.var_names).describe(percentiles=[0.5, 0.95, 0.99]).T.round(1)

#### Inspecting Visium patching 

In [ ]:
# Curious about PDPN, TCF7, CD14 and CD274 genes from the Badoual pre-print:
# https://www.biorxiv.org/content/10.1101/2024.10.03.616510v1.full.pdf

# Swap with whichever gene you want to view/what sample you want to look for it in
sc.pl.spatial(vis[vis.obs["sample"] == "S01_MAR21"], library_id="S01_MAR21", color="PDPN")

#### Checking top genes being expressed in MAR21

In [ ]:
mar21 = adatas["S01_MAR21"]
# Raw UMI counts, spots × genes (sparse)
counts = mar21.X

# General view of expression
gene_stats = pd.DataFrame(
    {
        "mean_UMI_per_spot": np.asarray(counts.mean(axis=0)).ravel(),
        "pct_spots_detected": np.asarray((counts > 0).mean(axis=0)).ravel() * 100,
    },
    index=mar21.var_names,
)
gene_stats["rank"] = gene_stats.mean_UMI_per_spot.rank(ascending=False).astype(int)

# display(gene_stats.sort_values("mean_UMI_per_spot", ascending=False).head(20).round(2))

# Looking at a subset of genes that have a Codex protein marker
genes = [
    "KRT5",
    "KRT14",
    "KRT13",
    "KRT8",
    "KRT18",
    "MKI67",
    "CD68",
    "HLA-DRA",
    "LAMP1",
    "PTPRC",
    "CD4",
    "CD3E",
    "CD8A",
    "MS4A1",
]
display(gene_stats.loc[genes].round(2))

#### Same for SEP21 

In [ ]:
sep21 = adatas["S01_SEP21"]
# Raw UMI counts, spots × genes (sparse)
counts = sep21.X

# General view of expression
gene_stats = pd.DataFrame(
    {
        "mean_UMI_per_spot": np.asarray(counts.mean(axis=0)).ravel(),
        "pct_spots_detected": np.asarray((counts > 0).mean(axis=0)).ravel() * 100,
    },
    index=sep21.var_names,
)
gene_stats["rank"] = gene_stats.mean_UMI_per_spot.rank(ascending=False).astype(int)

# display(gene_stats.sort_values("mean_UMI_per_spot", ascending=False).head(20).round(2))

# Looking at a subset of genes that have a Codex protein marker
genes = [
    "KRT5",
    "KRT14",
    "KRT13",
    "KRT8",
    "KRT18",
    "MKI67",
    "CD68",
    "HLA-DRA",
    "LAMP1",
    "PTPRC",
    "CD4",
    "CD3E",
    "CD8A",
    "MS4A1",
]
display(gene_stats.loc[genes].round(2))

# Much less mean UMI, maybe sequencing depth was worse? Would lead to noisier correlations/alignment
# Reading the SEP21_OPSCC_analysis.Rmd from the original github https://github.com/BiomedicalMachineLearning/HeadAndNeck_DeepCaseStudy
# SEP21 is just tumour, whereas MAR21 has healthy tissue - could explain different expression
# Going to preferentially try aligning MAR21 with CODEX

#### Checking Codex cell positions on the image 

In [ ]:
list(codex.var_names)
# short names
list(codex.var["panel_name"])

In [ ]:
tif = tifffile.TiffFile(CODEX / "Experiment_240522_Jaz(8_9).qptiff")
# print([lv.shape for lv in tif.series[0].levels])
# 4-level pyramid
img = tif.series[0].levels[3].asarray()
# smallest level: 12 × 1188 × 1416 - 4Mb only vs 711Mb for full image
um_per_px = 0.3774 * 8
# full-res px size × downsample factor


def stretch(x):
    lo, hi = np.percentile(x, [1, 99.5])
    return np.clip((x - lo) / (hi - lo), 0, 1)


fig, ax = plt.subplots(1, 2, figsize=(16, 7))
extent = [0, img.shape[2] * um_per_px, img.shape[1] * um_per_px, 0]  # draw image in µm

# RGB image with PanCK in red, DAPI in bklue
ax[0].imshow(np.dstack([stretch(img[6]), np.zeros(img.shape[1:]), stretch(img[0])]), extent=extent)
ax[0].scatter(*codex.obsm["spatial"][::20].T, s=0.5, c="yellow", alpha=0.5)
ax[0].set_title("Image: PanCK (red), DAPI (blue); every 20th cell centroid (yellow)")

# Per-cell mean PanCK - epithelial/tumour cells pale / lymphoid nodules darker
p99 = np.percentile(codex[:, "PANCK"].X, 99)
sc_plot = ax[1].scatter(
    *codex.obsm["spatial"].T, c=codex[:, "PANCK"].X.ravel(), s=0.2, cmap="magma", vmin=0, vmax=p99
)
fig.colorbar(sc_plot, ax=ax[1], label="PanCK intensity (Cell mean, 0–255 scale)", shrink=0.8)
ax[1].invert_yaxis()
ax[1].set_aspect("equal")
ax[1].set_title("Per-cell PanCK (Cell mean)")

In [ ]:
# Further resolution about PanCK+ tissues as being lymphoid or epithelial/tumour

pan = codex[:, "PANCK"].X.ravel()
is_pos = np.log1p(pan) > 1.2  # the low point between the two peaks in Cell F (≈ 2.3 raw)
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(
    *codex.obsm["spatial"][~is_pos].T, s=0.3, c="lightgrey", label=f"PanCK− ({(~is_pos).sum()})"
)
ax.scatter(*codex.obsm["spatial"][is_pos].T, s=0.3, c="crimson", label=f"PanCK+ ({is_pos.sum()})")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.legend(markerscale=20)

#### Checking intensity distribution per Codex marker

In [ ]:
# Looking for bimodality, where one big negative peak is there (background)
# with a separate peak or tail for positive cells (signal)

# Large CD8/CD3e counts from before are most likely background

df = pd.DataFrame(codex.X, columns=codex.var_names)

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, m in zip(axes.ravel(), codex.var_names):
    ax.hist(np.log1p(df[m]), bins=100, color="grey")
    ax.set_title(m)
    ax.set_xlabel("log1p(Cell mean)")
plt.tight_layout()

#### Codex marker spread on tissue 

In [ ]:
xy = codex.obsm["spatial"]
BIN = 25  # bin size in µm; try 55 to match the Visium spot size
xbins = np.arange(0, xy[:, 0].max() + BIN, BIN)
ybins = np.arange(0, xy[:, 1].max() + BIN, BIN)
extent = [xbins[0], xbins[-1], ybins[-1], ybins[0]]  # axes in µm, y pointing down like the image

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
for ax, m in zip(axes.ravel(), codex.var_names):
    grid = binned_statistic_2d(
        xy[:, 1],
        xy[:, 0],
        df[m],
        statistic="mean",  # averaging to smooth-out cell noise
        bins=[ybins, xbins],
    ).statistic
    im = ax.imshow(
        grid,
        extent=extent,
        cmap="magma",
        vmin=np.nanpercentile(grid, 1),
        vmax=np.nanpercentile(grid, 99),
    )
    ax.set_facecolor("lightgrey")  # empty bins (no cells) show as grey
    fig.colorbar(im, ax=ax, shrink=0.75, label="mean intensity")
    ax.set_title(m, fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

# CD68, CD45 and CD107a in upper right lobe show an immune/macrophage cell-rich centre

# CD4, CD45RO and HLA-DR up in bottom left lobe show CD4 T-cell rich centre
# Grid like pattern over CD45RO and HLA-DR is imaging artefact I think

# PanCK is high around epithelial/tumour sections but also moderate inside large round blobs
# Potentially these lobes are tumour nests growing inside tonsillar tissue surrounded by lymphoid tissue
# https://www.annualreviews.org/content/journals/10.1146/annurev-pathmechdis-031521-041424

#### Checking Codex markers co-occurrence

In [ ]:
# Correlation heatmap between markers
corr = df.drop(columns="DAPI").corr(method="spearman")
sns.clustermap(corr, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".2f", figsize=(9, 9))

# Sanity check: tumour cells (PanCK-high) should not carry immune markers
# PanCK+ cells should be CD8-
pan_hi = df["PANCK"] > df["PANCK"].quantile(0.90)
pan_lo = df["PANCK"] < df["PANCK"].quantile(0.50)
pd.DataFrame(
    {
        "median in PanCK top 10%": df[pan_hi].median(),
        "median in PanCK bottom 50%": df[pan_lo].median(),
        "p95 all cells": df.quantile(0.95),
    }
).round(1)

# CD8 signal is most likely background, pretty much same expression across all cells tumour/not
# CD20 (B cells) vs PanCK (epithelium) makes no sense, same for CD8 vs CD68 (macrophages)/ CD20
# Maybe signal spillover/autofluorescence from IHC?

# RNA-protein comparison might be able to do PanCK vs keratin (KRT5, KRT14, KRT17)?
# Keratin genes have highest RNA

### Placing CODEX cells on MAR21 Visium 

#### Scaling MAR21 H&E/Building H&E binary masks

In [ ]:
MAR = DATA / "Visium_S01_MAR21"
he = np.asarray(Image.open(MAR / "spatial/tissue_hires_image.png"))[..., :3]
sf = json.loads((MAR / "spatial/scalefactors_json.json").read_text())
um_per_fullres = 55 / sf["spot_diameter_fullres"]
# Visium spot is 55 µm across
um_per_hires = um_per_fullres / sf["tissue_hires_scalef"]
# print(f"Visium: {um_per_fullres:.3f} µm per full-res px, {um_per_hires:.3f} µm per hires px")
# Visium: 0.933 µm per full-res px, 3.278 µm per hires px

grey = ndi.gaussian_filter(he.mean(axis=2), 2)
vis_mask = grey < 200
# Tissue is darker than glass
frame = 110
# Blank out the fiducial dot frame
vis_mask[:frame] = vis_mask[-frame:] = False
vis_mask[:, :frame] = vis_mask[:, -frame:] = False
vis_mask = remove_small_objects(closing(vis_mask, disk(3)), max_size=300)

groups = label(dilation(vis_mask, disk(15)))
# Merging nearby fragments into pieces
props = sorted([r for r in regionprops(groups) if r.area > 20000], key=lambda r: -r.area)
pieces = {i + 1: r for i, r in enumerate(props)}
# Piece 1 = largest and so forth

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(he)
for i, r in pieces.items():
    y0, x0, y1, x1 = r.bbox
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="red", lw=2))
    ax.text(x0 + 15, y0 + 60, f"piece {i}", color="red", fontsize=14, weight="bold")

#### Loading in the MAR21 pathologist annotations 

In [ ]:
ann = pd.read_csv(DATA / "pathologist_annotations_MAR21.csv", index_col=0)
print(ann.shape)
display(ann.head())
print(ann["pathologist_annotation"].value_counts(dropna=False))

# Do the barcodes match the Visium spots?
print("annotated spots also in mar21:", ann.index.isin(mar21.obs_names).sum(), "/", len(ann))
print("mar21 spots without a label:  ", (~mar21.obs_names.isin(ann.index)).sum())

#### Attaching annotations to the Visium object + plot

In [ ]:
mar21.obs["pathologist"] = ann["pathologist_annotation"].reindex(mar21.obs_names).astype("category")
sc.pl.spatial(
    mar21,
    img_key="hires",
    library_id="S01_MAR21",
    color="pathologist",
    size=1.3,
    title="MAR21 pathologist annotations",
)

#### Sanity check: Checking which tissue pieces hold which labels 

In [ ]:
# Reading more about the paper, each piece of tissue of this sample is accounted for
# Piece 2 is tumour, hence Codex
# Piece 1 is likely a healthy control from the soft-palate

hires_xy = (mar21.obsm["spatial"] * sf["tissue_hires_scalef"]).round().astype(int)
# (x, y) in hires px
label_at_spot = groups[hires_xy[:, 1], hires_xy[:, 0]]
# Image indexed [row=y, col=x]
label_to_piece = {r.label: i for i, r in pieces.items()}
mar21.obs["piece"] = [
    f"piece {label_to_piece[l]}" if l in label_to_piece else "none" for l in label_at_spot
]
mar21.obs["piece"] = mar21.obs["piece"].astype("category")

pd.crosstab(mar21.obs["pathologist"], mar21.obs["piece"])

In [ ]:
# Looking at H&E, Visium and Pathologist annotations together
sc.pl.spatial(
    mar21,
    img_key="hires",
    library_id="S01_MAR21",
    color="pathologist",
    size=0.8,
    alpha=0.5,
    crop_coord=(100, 4200, 100, 2800),
)

#### Defining a Codex tissue mask as (x,y) points in µm scale

In [ ]:
img = (
    tifffile.TiffFile(DATA / "CODEX_S01/Experiment_240522_Jaz(8_9).qptiff")
    .series[0]
    .levels[3]
    .asarray()
)


# Stretch H&E high res images to fit
def stretch(x):
    lo, hi = np.percentile(x, [1, 99.5])
    return np.clip((x - lo) / (hi - lo), 0, 1)


um_per_codex_px = 0.3774 * 8
# Pyramid level 3
signal = ndi.gaussian_filter(stretch(img[0]) + stretch(img[6]), 4)
# DAPI + PanCK as strongest signal markers for epithelium/nuclei from checking Codex cell positions
codex_mask = signal > threshold_otsu(signal)
codex_mask = ndi.binary_fill_holes(
    remove_small_objects(opening(closing(codex_mask, disk(4)), disk(3)), max_size=400)
)
# Handling mask as points so can use matrix math to align it over cell centroids
rows, cols = np.nonzero(codex_mask)
mask_pts_um = np.column_stack([cols, rows]) * um_per_codex_px  # (x, y) in µm

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(signal, cmap="gray")
ax[0].set_title("DAPI + PanCK (smoothed)")
ax[1].imshow(codex_mask, cmap="gray")
ax[1].set_title("CODEX tissue mask")

#### Some helper functions to help manipulate (flip,rotate,shift) Codex mask 

In [ ]:
def rot_flip(pts_um, angle_deg, flip):
    """Mirror x (optional), then rotate about the origin. Units stay µm."""
    p = pts_um * np.array([-1, 1]) if flip else pts_um
    a = np.deg2rad(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return p @ R.T


def rasterise(pts_px):
    """(x, y) points in hires px → binary image, plus where its top-left corner sits."""
    origin = np.floor(pts_px.min(axis=0)).astype(int)
    ij = np.round(pts_px - origin).astype(int)
    im = np.zeros(ij.max(axis=0)[::-1] + 1, bool)
    im[ij[:, 1], ij[:, 0]] = True
    return closing(im, disk(1)), origin


def best_shift(piece_mask, codex_im):
    """Slide the CODEX mask over every position on the piece; return the best Dice and its position."""
    overlap = fftconvolve(piece_mask.astype(float), codex_im[::-1, ::-1].astype(float), mode="full")
    dice = 2 * overlap / (piece_mask.sum() + codex_im.sum())
    i, j = np.unravel_index(np.argmax(dice), dice.shape)
    return dice[i, j], np.array([j - codex_im.shape[1] + 1, i - codex_im.shape[0] + 1])  # (x, y)

#### Sanity Check: Matching up H&E and Codex masks

In [ ]:
# Checking 5 pieces x 2 flips x 120 angles = 1200 different orientations
# Piece 2 of the H&E lines up very well with the Codex
# Will go with this subsection for alignment

results = []
for pid, r in pieces.items():
    y0, x0, y1, x1 = r.bbox
    piece_mask = (groups[y0:y1, x0:x1] == r.label) & vis_mask[y0:y1, x0:x1]
    for flip in [False, True]:
        for angle in range(0, 360, 3):
            codex_im, origin = rasterise(rot_flip(mask_pts_um, angle, flip) / um_per_hires)
            d, tl = best_shift(piece_mask, codex_im)
            results.append(
                {
                    "piece": pid,
                    "flip": flip,
                    "angle": angle,
                    "dice": d,
                    "shift": tl + np.array([x0, y0]) - origin,
                }
            )
results = pd.DataFrame(results)
best_per_piece = results.loc[results.groupby("piece").dice.idxmax()].sort_values(
    "dice", ascending=False
)
display(best_per_piece[["piece", "flip", "angle", "dice"]].round(3))

# Does not tell me if the contents of the masks agree (epithelium vs epithelium, etc.)
top = best_per_piece.iloc[0]
fig, ax = plt.subplots(figsize=(12, 4))
for (pid, flip), g in results.groupby(["piece", "flip"]):
    ax.plot(g.angle, g.dice, lw=2.5 if pid == top.piece else 0.7, label=f"piece {pid}, flip={flip}")
ax.set_xlabel("rotation (°)")
ax.set_ylabel("best Dice")
ax.legend(fontsize=8, ncol=2)

#### Sanity check: Do the H&E/Codex contents match up (PanCK vs keratin genes)?

In [ ]:
KERATINS = ["KRT5", "KRT14", "KRT13"]

# Normalize keratin RNA score to Visium spot size
norm = sc.pp.normalize_total(mar21, target_sum=1e4, inplace=False)["X"]
# per 10k UMIs; mar21 itself unchanged
krt_score = np.log1p(np.asarray(norm[:, mar21.var_names.get_indexer(KERATINS)].sum(axis=1)).ravel())
# print("keratin score range:", krt_score.min().round(2), krt_score.max().round(2))
# Keratin score range: 4.03 - 7.15

##### Sanity Check: Convert Codex cells into pseudo-spots

In [ ]:
# Helper functions

spot_tree = cKDTree(mar21.obsm["spatial"])
radius_px = 27.5 / um_per_fullres
# Spot radius (27.5 µm) in full-res px
panck = codex[:, "PANCK"].X.ravel()


def to_fullres(xy_um, angle, flip, shift_hires):
    return (rot_flip(xy_um, angle, flip) / um_per_hires + shift_hires) / sf["tissue_hires_scalef"]


def pseudo_spots(angle, flip, shift_hires, min_cells=3):
    """Mean CODEX PanCK per Visium spot, from cells within 27.5 µm of the spot centre."""
    xy = to_fullres(codex.obsm["spatial"], angle, flip, shift_hires)
    dist, spot_idx = spot_tree.query(xy, distance_upper_bound=radius_px)
    inside = np.isfinite(dist)
    # Cells outside every spot get dist = inference
    n = np.bincount(spot_idx[inside], minlength=mar21.n_obs)
    total = np.bincount(spot_idx[inside], weights=panck[inside], minlength=mar21.n_obs)
    keep = n >= min_cells
    return keep, total[keep] / n[keep], n


def score(angle, flip, shift_hires):
    keep, panck_mean, _n = pseudo_spots(angle, flip, shift_hires)
    if keep.sum() < 50:
        return np.nan, keep.sum()
    return spearmanr(panck_mean, krt_score[keep]).statistic, keep.sum()

#### Sanity check: Correlation for every orientation from H&E/Codex matchup

In [ ]:
rho, nspots = zip(*[score(r.angle, r.flip, np.asarray(r.shift)) for r in results.itertuples()])
results["rho"], results["n_spots"] = rho, nspots

display(
    results.loc[
        results.groupby("piece").dice.idxmax(), ["piece", "flip", "angle", "dice", "rho", "n_spots"]
    ].round(3)
)
display(
    results.sort_values("rho", ascending=False)
    .head(10)[["piece", "flip", "angle", "dice", "rho", "n_spots"]]
    .round(3)
)
print(
    "ρ across all orientations: median",
    np.nanmedian(results.rho).round(3),
    "| 95th percentile",
    np.nanpercentile(results.rho, 95).round(3),
)

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
for (pid, flip), g in results.groupby(["piece", "flip"]):
    ax[0].plot(
        g.angle, g.rho, lw=2.5 if pid == top.piece else 0.7, label=f"piece {pid}, flip={flip}"
    )
ax[0].axhline(0, c="k", lw=0.5)
ax[0].set_xlabel("rotation (°)")
ax[0].set_ylabel("Spearman ρ, PanCK vs keratin RNA")
ax[0].legend(fontsize=7, ncol=2)
ax[1].scatter(results.dice, results.rho, s=6, c=results.piece, cmap="tab10")
ax[1].set_xlabel("outline Dice")
ax[1].set_ylabel("content ρ")
ax[1].set_title("Does a better outline fit mean better content agreement?")

#### Sanity Check: Viewing the best-by-content alignment

In [ ]:
# Rigid alignment does not put CODEX cells over the matching tissue (PanCK vs KRT ρ ≈ 0.03)
# Does not say that piece 2 and CODEX differ in content, might need further custom fitting

best = results.loc[results.rho.idxmax()]
keep, panck_mean, n = pseudo_spots(best.angle, best.flip, np.asarray(best["shift"]))
xy_spot = mar21.obsm["spatial"][keep] * sf["tissue_hires_scalef"]

fig, ax = plt.subplots(1, 3, figsize=(24, 8))
for a in ax:
    a.imshow(he)
    a.set_xlim(xy_spot[:, 0].min() - 80, xy_spot[:, 0].max() + 80)
    a.set_ylim(xy_spot[:, 1].max() + 80, xy_spot[:, 1].min() - 80)
p = ax[1].scatter(*xy_spot.T, c=krt_score[keep], s=12, cmap="magma")
fig.colorbar(p, ax=ax[1], shrink=0.7)
ax[1].set_title("Visium keratin RNA (log1p per 10k)")
p = ax[2].scatter(*xy_spot.T, c=panck_mean, s=12, cmap="magma", vmax=np.percentile(panck_mean, 99))
fig.colorbar(p, ax=ax[2], shrink=0.7)
ax[2].set_title(
    f"CODEX PanCK per pseudo-spot (piece {best.piece}, rot {best.angle}°, flip={best.flip}, ρ={best.rho:.2f})"
)

#### Sanity Check: Trying custom Codex/H&E alignment for tissue piece 2 using SimpleITK

In [ ]:
# If CODEX is placed on the H&E at a given rotation,
# do the CODEX tumour cells land on the spots the pathologist called tumour?

in_p2 = (mar21.obs["piece"] == "piece 2").to_numpy()
spot_um = mar21.obsm["spatial"][in_p2] * um_per_fullres
# Spot centres in µm
lab = mar21.obs["pathologist"].to_numpy()[in_p2]
pan_pos = np.log1p(codex[:, "PANCK"].X.ravel()) > 1.2


# Pick a carcinoma + one lymphocyte spot at random,
# AUC score shows chance a carcinoma spot has a higher PanCK+ fraction
def auc_for(angle, flip, shift_hires, radius_um=27.5):
    xy = (
        rot_flip(codex.obsm["spatial"], angle, flip) + np.asarray(shift_hires) * um_per_hires
    )  # CODEX cells in Visium µm
    hits = cKDTree(xy).query_ball_point(spot_um, r=radius_um)
    # Find all Codex cells within a Visium spot based on location (shift in x/y)
    n = np.array([len(h) for h in hits])
    # What fraction og cells are PanCK+
    frac = np.array([pan_pos[h].mean() if len(h) else np.nan for h in hits])
    car = (n >= 3) & (lab == "Invasive carcinoma")
    lym = (n >= 3) & (lab == "Lymphocytes")
    if car.sum() < 10 or lym.sum() < 10:
        return np.nan
    return roc_auc_score(
        np.r_[np.ones(car.sum()), np.zeros(lym.sum())], np.r_[frac[car], frac[lym]]
    )


p2 = results[results.piece == 2].copy()
p2["auc"] = [auc_for(r.angle, r.flip, r.shift) for r in p2.itertuples()]
print("AUC over all orientations: median", round(np.nanmedian(p2.auc), 3))
display(p2.sort_values("auc", ascending=False).head(8)[["flip", "angle", "dice", "auc"]].round(3))

# AUC peaks near 204°, flip=False at the outline-fitted position (0.84 vs ~0.5 for other angles)
# That being said: with free shifting the mirrored 345° flip also reaches ~0.95, so the orientation not yet settled
fig, ax = plt.subplots(figsize=(12, 4))
for flip, g in p2.groupby("flip"):
    ax.plot(g.angle, g.auc, label=f"flip={flip}")
ax.axhline(0.5, c="k", lw=0.5)
ax.set_xlabel("rotation (°)")
ax.set_ylabel("AUC: carcinoma vs lymphocyte")
ax.legend()

#### Sanity check: Haematoxylin (H&E nuclei) vs CODEX cell density per spot

In [ ]:
spots_p2 = mar21.obsm["spatial"][in_p2] * um_per_fullres  # piece-2 spot centres, µm
lab = mar21.obs["pathologist"].to_numpy()[in_p2]
umi = np.asarray(mar21.X[in_p2].sum(axis=1)).ravel()  # total UMIs per spot

hema = rgb2hed(he[..., :3])[..., 0]  # haematoxylin (nuclear stain) channel
yy, xx = np.mgrid[: hema.shape[0], : hema.shape[1]]
pix_tree = cKDTree(
    np.column_stack([xx.ravel(), yy.ravel()]) * um_per_hires
)  # every H&E pixel, in µm
hema_spot = np.array([hema.ravel()[h].mean() for h in pix_tree.query_ball_point(spots_p2, r=27.5)])


def codex_cells_per_spot(angle, flip, shift_hires, dx=0, dy=0):
    """Place CODEX (rigid), count cells within 27.5 µm of each piece-2 spot."""
    xy = (
        rot_flip(codex.obsm["spatial"], angle, flip)
        + np.asarray(shift_hires) * um_per_hires
        + [dx, dy]
    )
    return cKDTree(xy).query_ball_point(spots_p2, r=27.5, return_length=True)


row204 = results[(results.piece == 2) & (results.angle == 204) & (~results.flip)].iloc[0]
n204 = codex_cells_per_spot(204, False, row204["shift"])
table = pd.DataFrame(
    {"pathologist": lab, "UMI": umi, "haematoxylin": hema_spot, "CODEX_cells": n204}
)
print(table.groupby("pathologist").median().round(3))
print(
    "\nSpearman, same section  UMI vs haematoxylin:", round(spearmanr(umi, hema_spot).statistic, 3)
)
print("Spearman at 204°        CODEX cells vs UMI:", round(spearmanr(n204, umi).statistic, 3))
print(
    "Spearman at 204°        CODEX cells vs haematoxylin:",
    round(spearmanr(n204, hema_spot).statistic, 3),
)

#### Sanity Check: Correlating haematoxylin across every piece-2 orientation

In [ ]:
p2 = results[results.piece == 2].copy()
p2["rho_hema"] = [
    spearmanr(codex_cells_per_spot(r.angle, r.flip, r.shift), hema_spot).statistic
    for r in p2.itertuples()
]
print(
    p2.sort_values("rho_hema", ascending=False)[["angle", "flip", "dice", "rho_hema"]]
    .head(8)
    .round(3)
    .to_string(index=False)
)
print(
    "null: median", p2.rho_hema.median().round(3), " 95th pct", p2.rho_hema.quantile(0.95).round(3)
)

fig, ax = plt.subplots(figsize=(10, 4))
for flip, g in p2.groupby("flip"):
    ax.plot(g.angle, g.rho_hema, label=f"flip={flip}")
ax.axhline(p2.rho_hema.quantile(0.95), c="grey", ls="--", label="95th pct of all placements")
ax.set_xlabel("rotation (°)")
ax.set_ylabel("ρ, CODEX cells vs H&E haematoxylin")
ax.legend()
plt.show()

#### Sanity check: Sweeping degree offset around 204° and 345° (flip)

In [ ]:
offsets = np.arange(-300, 301, 25)
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for a, (ang, fl) in zip(ax, [(204, False), (345, True)]):
    row = p2[(p2.angle == ang) & (p2.flip == fl)].iloc[0]
    S = np.array(
        [
            [
                spearmanr(codex_cells_per_spot(ang, fl, row["shift"], dx, dy), hema_spot).statistic
                for dx in offsets
            ]
            for dy in offsets
        ]
    )
    i, j = np.unravel_index(np.nanargmax(S), S.shape)
    print(
        f"{ang}° flip={fl}: ρ at zero shift {S[12, 12]:.3f}; peak {S[i, j]:.3f} at dx={offsets[j]}, dy={offsets[i]} µm"
    )
    im = a.imshow(S, extent=[-312.5, 312.5, 312.5, -312.5], cmap="viridis", vmin=-0.15, vmax=0.3)
    a.plot(0, 0, "w+", ms=15, mew=2)
    a.plot(offsets[j], offsets[i], "rx", ms=12, mew=2)
    a.set_title(f"{ang}°, flip={fl}")
    a.set_xlabel("x shift (µm)")
    a.set_ylabel("y shift (µm)")
    fig.colorbar(im, ax=a, label="ρ, CODEX cells vs haematoxylin")
plt.show()

### Image-based registration for Codex DAPI vs H&E haematoxylin

In [ ]:
SIG = 2
# Light smoothing (~6.5 µm)
r2 = pieces[2]
y0, x0, y1, x1 = r2.bbox
pad = 60
Y0, X0, Y1, X1 = (
    max(y0 - pad, 0),
    max(x0 - pad, 0),
    min(y1 + pad, he.shape[0]),
    min(x1 + pad, he.shape[1]),
)
fixed = ndi.gaussian_filter(rgb2hed(he[..., :3])[..., 0], SIG)[Y0:Y1, X0:X1]
# H&E haematoxylin = nuclei
fixed_mask = dilation(groups == r2.label, disk(15))[Y0:Y1, X0:X1]
# Piece 2 only
dapi_s = ndi.gaussian_filter(stretch(img[0]), SIG)
# Codex DAPI = nuclei (level 3)
cmask = codex_mask.astype(float)


def warp_codex(image, angle, shift_hires, order=1):
    """Resample a CODEX level-3 image into the H&E crop, same convention as rot_flip (no flip)."""
    a = np.deg2rad(angle)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])

    def inv(xy):  # (col,row) in H&E crop → (col,row) in CODEX image
        q_um = (xy + [X0, Y0] - np.asarray(shift_hires)) * um_per_hires
        return (q_um @ R) / um_per_codex_px

    return warp(image, inv, output_shape=fixed.shape, order=order, cval=0)

#### Sanity Check: Rotating images with rigid fit

In [ ]:
s0 = np.asarray(
    results[(results.piece == 2) & (results.angle == 204) & (~results.flip)].iloc[0]["shift"], float
)
c_um = mask_pts_um.mean(axis=0)


# Codex tissue centre, µm
def shift_for(angle):
    # Rotate about the tissue centre
    return (
        s0
        + (rot_flip(c_um[None], 204, False)[0] - rot_flip(c_um[None], angle, False)[0])
        / um_per_hires
    )


def best_shift_ncc(angle, region, max_px=60):
    """NCC for every shift up to ±max_px (±197 µm); return (best NCC, shift in hires px)."""
    s = shift_for(angle)
    mov = warp_codex(dapi_s, angle, s)
    mmask = (warp_codex(cmask, angle, s, order=0) > 0.5) & region
    C = cross_correlate_masked(fixed, mov, region, mmask, mode="full", overlap_ratio=0.3)
    c0 = np.array(fixed.shape) - 1
    # zero shift sits here in "full" mode
    W = C[c0[0] - max_px : c0[0] + max_px + 1, c0[1] - max_px : c0[1] + max_px + 1]
    i, j = np.unravel_index(np.argmax(W), W.shape)
    return W[i, j], s + [j - max_px, i - max_px]


def register(region, angles=None):
    if angles is None:
        angles = np.arange(198, 220.01, 0.5)
    scores = [(*best_shift_ncc(a, region), a) for a in angles]
    ncc, shift, angle = max(scores, key=lambda t: t[0])
    return {"ncc": ncc, "angle": angle, "shift": shift, "curve": [(a, n) for n, _, a in scores]}


fit = register(fixed_mask)
print(
    f"global image fit: angle {fit['angle']}°, shift {fit['shift'].round(1)} hires px, NCC {fit['ncc']:.3f}"
)
moved = (fit["shift"] - shift_for(fit["angle"])) * um_per_hires
# How far the CODEX tissue centre moved vs the outline fit
print(f"tissue centre moved by (x, y) = {moved.round(0)} µm vs the outline fit")
plt.plot(*zip(*fit["curve"]))
plt.xlabel("rotation (°)")
plt.ylabel("best NCC over shifts")
plt.show()

##### Sanity Check: Overlaying H&E (pink) with Codex DAPI (green) with agreement in white 

In [ ]:
def overlay(angle, shift_hires):
    mov = warp_codex(dapi_s, angle, shift_hires)
    f = np.clip(fixed / np.percentile(fixed[fixed_mask], 99), 0, 1)
    m = np.clip(mov / np.percentile(mov[mov > 0], 99), 0, 1)
    return np.dstack([f, m, f])


fig, ax = plt.subplots(1, 2, figsize=(24, 9))
ax[0].imshow(overlay(204, s0))
ax[0].set_title("outline fit (R4), 204°")
ax[1].imshow(overlay(fit["angle"], fit["shift"]))
ax[1].set_title(f"image fit, {fit['angle']}°")
plt.show()